## Leer datos y verificar contenido

Descargamos datos, los cargamos en dataframe y mostramos por pantalla cinco filas y el nombre de las columnas

In [ ]:
!wget -O wine.xlsx https://github.com/jordipereiragude/dataforcourses/raw/refs/heads/main/wine.xlsx

In [ ]:
import pandas

df=pandas.read_excel("wine.xlsx")
print("=======")
print(df.head())
print("=======")
print(df.dtypes)
print("=======")



## Visualización básica de relaciones

Haz un gráfico de dispersión mostrando en el eje X el valor de la columna AGST y en el eje Y el valor de la columna Price

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(df['AGST'], df['Price'], color='blue', alpha=0.7)
plt.title('Gráfico de Dispersión: AGST vs Price')
plt.xlabel('AGST (Average Growing Season Temperature)')
plt.ylabel('Price')
plt.grid(True)
plt.show()

La generación de nuevos gráficos puede basarse en los valores de gráficos anteriores.

Por ejemplo, podemos modificar el gráfico anterior para representar la relación con `Age`

## Modelo básico

Construye un modelo de regresión lineal utilizando statmodels y su interface de fórmulas. La fórmula del modelo es:

Price ~ AGST

Al acabar, mostrar un resumen del modelo usando summary()

In [ ]:
import statsmodels.formula.api as smf

# Construir y ajustar el modelo de regresión lineal
modelo = smf.ols(formula='Price ~ AGST', data=df).fit()

# Mostrar el resumen del modelo
print(modelo.summary())

## Modelo con todas las variables

Construiremos un modelo:

Price ~ AGST + Year + WinterRain + HarvestRain + Age

In [ ]:
import statsmodels.formula.api as smf

# Construir y ajustar el modelo de regresión lineal
modelo = smf.ols(formula='Price ~ AGST + Year + WinterRain + HarvestRain + Age', data=df).fit()

# Mostrar el resumen del modelo
print(modelo.summary())

## Verificación de resultados

Construye una tabla de correlación entre las variables numéricas del dataframe. Usa un "heatmap" para colorear las relaciones

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calcular la matriz de correlación de las variables numéricas
corr_matrix = df.corr()

# Crear el mapa de calor (heatmap)
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlación de las Variables')
plt.show()

Modifiquemos el modelo. Hay que alterar la línea de código:

```
modelo = smf.ols(formula='Price ~ AGST + Year + WinterRain + HarvestRain', data=df).fit()
```

In [ ]:
import statsmodels.formula.api as smf

# Construir y ajustar el modelo de regresión lineal
modelo = smf.ols(formula='Price ~ AGST + Year + WinterRain + HarvestRain', data=df).fit()

# Mostrar el resumen del modelo
print(modelo.summary())

## Verificar columnas y variables

Reporta el VIF de las diferentes variables del modelo

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import statsmodels.api as sm

# Seleccionar las variables explicativas del último modelo
variables = ['AGST', 'Year', 'WinterRain', 'HarvestRain']
X = df[variables]

# Añadir una constante para el cálculo del VIF
X = sm.add_constant(X)

# Calcular el VIF para cada variable explicativa
vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# Mostrar los resultados (excluyendo la constante)
print(vif_data[vif_data['Variable'] != 'const'])

Usando el modelo

Price ~ AGST

reporta la distancia de Cook de cada observación. Presenta un plot de la distancia de Cook de cada observación seguido de una tabla con las observaciones más influyentes. En la tabla reporta también el valor pronosticado por el modelo

In [ ]:
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import pandas as pd

# 1. Ajustar el modelo Price ~ AGST
modelo_simple = smf.ols(formula='Price ~ AGST', data=df).fit()

# 2. Calcular la distancia de Cook
influence = modelo_simple.get_influence()
cooks_d, p_values = influence.cooks_distance

# Añadir las distancias de Cook y los valores pronosticados al DataFrame
df_cook = df.copy()
df_cook['Cooks_Distance'] = cooks_d
df_cook['Price_Predicted'] = modelo_simple.predict(df)

# 3. Presentar un gráfico de la distancia de Cook
plt.figure(figsize=(10, 5))
plt.bar(df_cook.index, df_cook['Cooks_Distance'], color='crimson', alpha=0.8)
plt.axhline(y=4/len(df), color='blue', linestyle='--', label=f'Umbral (4/n = {4/len(df):.3f})')
plt.title('Distancia de Cook de cada observación')
plt.xlabel('Índice de la observación')
plt.ylabel('Distancia de Cook')
plt.legend()
plt.grid(True, axis='y')
plt.show()

# 4. Tabla con las observaciones más influyentes (que superan el umbral 4/n) incluyendo el valor pronosticado
umbral = 4 / len(df)
observaciones_influyentes = df_cook[df_cook['Cooks_Distance'] > umbral].sort_values(by='Cooks_Distance', ascending=False)

print("=== Tabla de Observaciones más Influyentes (Distancia de Cook > 4/n) ===")
print(observaciones_influyentes[['Year', 'Price', 'Price_Predicted', 'AGST', 'Cooks_Distance']].to_markdown())

## Análisis de residuos

Considera el modelo:

Price ~ AGST + WinterRain + HarvestRain + Age

crea un gráfico Q-Q

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Ajustar el modelo Price ~ AGST + WinterRain + HarvestRain + Age
formula_modelo = 'Price ~ AGST + WinterRain + HarvestRain + Age'
modelo_completo = smf.ols(formula=formula_modelo, data=df).fit()

# Obtener los residuos del modelo
residuos = modelo_completo.resid

# Crear el gráfico Q-Q con los textos traducidos al español
fig = sm.qqplot(residuos, line='s', fit=True)
plt.title(f'Gráfico Q-Q de los Residuos\nModelo: {formula_modelo}', fontsize=12)
plt.xlabel('Cuantiles Teóricos')
plt.ylabel('Cuantiles Muestrales')
plt.grid(True)
plt.show()

Crea gráficos que presentan cada variable (eje X) con los residuos del modelo (eje Y). Encima de cada gráfico pon también una estimación de la curva o recta que se estima que siguen los residuos

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np

# Variables independientes del modelo
variables_modelo = ['AGST', 'WinterRain', 'HarvestRain', 'Age']
formula_modelo = 'Price ~ AGST + WinterRain + HarvestRain + Age'

# Crear una cuadrícula de gráficos (2x2)
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))
fig.suptitle(f'Residuos vs Variables Explicativas\nModelo: {formula_modelo}', fontsize=16, y=0.96)

for i, var in enumerate(variables_modelo):
    row = i // 2
    col = i % 2
    ax = axes[row, col]

    x_vals = df[var]
    y_vals = residuos

    # Gráfico de dispersión
    ax.scatter(x_vals, y_vals, color='purple', alpha=0.7, edgecolors='k', label='Residuos')
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=1)

    # Ajustar y graficar una curva de tendencia no paramétrica local (LOWESS)
    lowess_fit = sm.nonparametric.lowess(y_vals, x_vals, frac=0.6)
    ax.plot(lowess_fit[:, 0], lowess_fit[:, 1], color='red', linewidth=2, label='Tendencia (LOWESS)')

    ax.set_title(f'Residuos vs {var}')
    ax.set_xlabel(var)
    ax.set_ylabel('Residuos')
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend()

plt.tight_layout(rect=[0, 0.03, 1, 0.92])
plt.show()

## Resolución de problemas de regresión

Crear nueva variable logPrice que incluya el logaritmo de la columna Price

In [ ]:
import numpy as np

# Crear la nueva variable logPrice con el logaritmo natural de Price
df['logPrice'] = np.log(df['Price'])

# Mostrar las primeras filas para verificar el cambio
print(df[['Price', 'logPrice']].head())

Modelo:

logPrice ~ AGST + WinterRain + HarvestRain + Age


In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt

modeloLog = smf.ols(formula='logPrice ~ AGST + WinterRain + HarvestRain + Age', data=df).fit()

print(modeloLog.summary())

## Wrap-up

Reportar error medio de

modelo_completo = smf.ols(formula='Price ~ AGST + WinterRain + HarvestRain + Age', data=df).fit()

y

modeloLog = smf.ols(formula='logPrice ~ AGST + WinterRain + HarvestRain + Age', data=df).fit()

considerando variable Price (por tanto en modeloLog hay que corregir los valores estimados)

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

# 1. Ajustar el modelo completo en la escala original
modelo_completo = smf.ols(formula='Price ~ AGST + WinterRain + HarvestRain + Age', data=df).fit()

# 2. Ajustar el modelo con la variable logarítmica
modeloLog = smf.ols(formula='logPrice ~ AGST + WinterRain + HarvestRain + Age', data=df).fit()

# 3. Obtener las predicciones en la escala original de Price
pred_completo = modelo_completo.predict(df)

# Para el modelo logarítmico, aplicamos la función exponencial para volver a la escala original
# Opcionalmente se puede aplicar la corrección por sesgo multiplicando por exp(varianza_residual / 2),
# pero calcularemos la conversión directa simple y la corregida para comparar.
pred_log_directa = np.exp(modeloLog.predict(df))

sigma2 = modeloLog.scale  # Varianza estimada de los residuos en el modelo logarítmico
pred_log_corregida = np.exp(modeloLog.predict(df) + (sigma2 / 2))

# Valores reales de Price
y_real = df['Price']

# 4. Calcular el Error Absoluto Medio (MAE)
mae_completo = np.mean(np.abs(y_real - pred_completo))
mae_log_directa = np.mean(np.abs(y_real - pred_log_directa))
mae_log_corregida = np.mean(np.abs(y_real - pred_log_corregida))

# Calcular el Error Cuadrático Medio (RMSE)
rmse_completo = np.sqrt(np.mean((y_real - pred_completo)**2))
rmse_log_directa = np.sqrt(np.mean((y_real - pred_log_directa)**2))
rmse_log_corregida = np.sqrt(np.mean((y_real - pred_log_corregida)**2))

# Reportar resultados
print("=== COMPARATIVA DE ERRORES MEDIOS (Variable objetivo: Price) ===\n")
print(f"1. Modelo Lineal Completo (Fórmula clásica):")
print(f"   - Error Absoluto Medio (MAE): {mae_completo:.2f}")
print(f"   - Raíz del Error Cuadrático Medio (RMSE): {rmse_completo:.2f}\n")
print(f"2. Modelo Logarítmico (Sin corrección de sesgo - exp(y_hat)):")
print(f"   - Error Absoluto Medio (MAE): {mae_log_directa:.2f}")
print(f"   - Raíz del Error Cuadrático Medio (RMSE): {rmse_log_directa:.2f}\n")
print(f"3. Modelo Logarítmico (Con corrección de sesgo - exp(y_hat + sigma^2 / 2)):")
print(f"   - Error Absoluto Medio (MAE): {mae_log_corregida:.2f}")
print(f"   - Raíz del Error Cuadrático Medio (RMSE): {rmse_log_corregida:.2f}")